# Gaussian information order and its non-Gaussian boundary

[Formal argument](../03_acquisition_information_monotonicity.md). Keep the original Gaussian positive control and add the correct posterior-averaging relation.

In [ ]:
import numpy as np
from hse_laplace.acquisition import gaussian_information_statistics, gaussian_posterior, loewner_margin
low_A = np.array([[1., 0.]])
high_A = np.array([[1., 0.], [0., 2.]])
low_s = gaussian_information_statistics(low_A, np.eye(1), np.array([0.2]))
high_s = gaussian_information_statistics(high_A, np.eye(2), np.array([0.2, -0.8]))
low_p = gaussian_posterior(np.zeros(2), np.eye(2), low_s)
high_p = gaussian_posterior(np.zeros(2), np.eye(2), high_s)
assert loewner_margin(high_s.information, low_s.information) >= -1e-12
assert loewner_margin(low_p.covariance, high_p.covariance) >= -1e-12
assert high_p.entropy() < low_p.entropy()
print('Gaussian posterior variances:', np.diag(low_p.covariance), np.diag(high_p.covariance))

## More information can increase a realized variance
Take $O_H=|\Theta|$ and constant $O_L$, so the degradation assumption is satisfied. The high-information variance equals 1 on a rare event, although its average is only 0.1.

In [ ]:
theta = np.array([-1., 0., 1.])
prior = np.array([.05, .90, .05])
var_low = prior @ theta**2 - (prior @ theta)**2
post_nonzero = prior[[0, 2]] / prior[[0, 2]].sum()
var_high_nonzero = post_nonzero @ theta[[0, 2]]**2
avg_var_high = .1 * var_high_nonzero + .9 * 0.
assert var_high_nonzero > var_low
np.testing.assert_allclose(avg_var_high, var_low)
print('Low / realized high / averaged high variances:', var_low, var_high_nonzero, avg_var_high)

## Enumerate a genuine noisy degradation channel
Balanced binary state, 90%-accurate high observation, then an 80%-accurate copy of that observation. Compute the joint law, rather than equating individual paired posteriors.

In [ ]:
joint = np.zeros((2, 2, 2))  # state, high observation, low observation
for state in range(2):
    for high in range(2):
        for low in range(2):
            joint[state, high, low] = .5 * (.9 if high == state else .1) * (.8 if low == high else .2)
post_high = joint.sum(axis=2)[1] / joint.sum(axis=(0, 2))
post_low = joint.sum(axis=1)[1] / joint.sum(axis=(0, 1))
for low in range(2):
    high_given_low = joint[:, :, low].sum(axis=0) / joint[:, :, low].sum()
    averaged = high_given_low @ post_high
    np.testing.assert_allclose(averaged, post_low[low])
np.testing.assert_allclose(post_low[1], .74)
assert not np.isclose(post_low[1], post_high[1])
print('P(state=1 | high=1):', post_high[1])
print('P(state=1 | low=1):', post_low[1])
print('Duplicate-noise variance: correct', 1/2, 'wrong independence', 1/3)

**Conclusion.** Retain Loewner monotonicity for its Gaussian assumptions. For general posteriors use conditional averaging, not a per-event width penalty. Independent latent-event views and degraded copies have different joint noise models.

In [ ]:
print("THEORY_DEMO_PASS::03_acquisition_information_monotonicity")